In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from torchvision import transforms
from tqdm import tqdm

1.8.0


In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN for binary classification of cats vs dogs."""

    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()

        # Convolutional Block 1
        # Input: (batch_size, 3, H, W) - RGB images
        # Output: (batch_size, 32, H/2, W/2) - 32 feature maps, half the spatial size
        self.conv_block1 = nn.Sequential(
            # First conv: 3 input channels (RGB) → 32 output feature maps
            # kernel_size=3: uses 3x3 filters, padding=1: maintains spatial dimensions
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # Normalizes the 32 feature maps for stable training
            nn.ReLU(),  # Non-linear activation function
            # Second conv: 32 → 32 feature maps, learns more complex patterns
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # Downsample by 2x: reduces spatial dimensions (e.g., 224x224 → 112x112)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Convolutional Block 2
        # Input: (batch_size, 32, H/2, W/2)
        # Output: (batch_size, 64, H/4, W/4) - doubles feature maps, halves spatial size
        self.conv_block2 = nn.Sequential(
            # Increase depth: 32 → 64 feature maps for learning more complex features
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Second conv at this depth level
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Downsample again: (e.g., 112x112 → 56x56)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Convolutional Block 3
        # Input: (batch_size, 64, H/4, W/4)
        # Output: (batch_size, 128, 4, 4) - highest-level features, fixed 4x4 spatial size
        self.conv_block3 = nn.Sequential(
            # Increase depth: 64 → 128 feature maps for high-level feature learning
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Second conv at this depth level
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Final spatial downsampling (e.g., 56x56 → 28x28)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Adaptive pooling: converts any spatial size to fixed 4x4
            # This makes the network flexible to different input image sizes
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        # Fully Connected Layers (Classifier)
        # Input: (batch_size, 128, 4, 4) = flattened to (batch_size, 2048)
        # Output: (batch_size, 2) - raw scores for cat and dog classes
        self.classifier = nn.Sequential(
            # Flatten 3D feature maps into 1D vector: 128×4×4 = 2048 features
            nn.Flatten(),
            # First dense layer: 2048 → 512 neurons
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),  # Randomly drop 50% of neurons during training to prevent overfitting
            # Second dense layer: 512 → 256 neurons
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),  # Another dropout layer for regularization
            # Output layer: 256 → 2 class scores (logits for cat and dog)
            nn.Linear(256, 2),
        )

    def forward(self, x):
        """
        Forward pass through the network.

        Args:
            x: Input tensor of shape (batch_size, 3, H, W)

        Returns:
            Output tensor of shape (batch_size, 2) containing class logits
        """
        # Pass through convolutional blocks to extract features
        x = self.conv_block1(x)  # Extract low-level features (edges, textures)
        x = self.conv_block2(x)  # Extract mid-level features (shapes, patterns)
        x = self.conv_block3(x)  # Extract high-level features (animal parts, structures)

        # Pass through classifier to get final predictions
        x = self.classifier(x)  # Convert features to class scores

        return x  # Returns raw logits (use with CrossEntropyLoss or apply softmax for probabilities)

In [38]:
class CatsDogsDataset(Dataset):
    """PyTorch Dataset for Cats vs Dogs."""

    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Convert numpy array to PIL Image if needed
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image.astype("uint8"))

        if self.transform:
            image = self.transform(image)

        return image, label

In [39]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [40]:
def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [41]:
def get_predictions(model, dataloader, device):
    """Get all predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)

In [42]:
def plot_training_history(history):
    """Plot training and validation loss and accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot loss
    ax1.plot(history["train_loss"], label="Train Loss", marker="o", linewidth=2)
    ax1.plot(history["val_loss"], label="Validation Loss", marker="o", linewidth=2)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("Training and Validation Loss", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Plot accuracy
    ax2.plot(history["train_acc"], label="Train Accuracy", marker="o", linewidth=2)
    ax2.plot(history["val_acc"], label="Validation Accuracy", marker="o", linewidth=2)
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("Accuracy (%)", fontsize=12)
    ax2.set_title("Training and Validation Accuracy", fontsize=14, fontweight="bold")
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✓ Training history plot saved as 'training_history.png'\n")

In [43]:
def plot_confusion_matrix(y_true, y_pred, classes=["Cat (0)", "Dog (1)"]):
    """Plot confusion matrix using only matplotlib."""
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))

    # Create heatmap using imshow
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count", rotation=270, labelpad=20)

    # Set ticks and labels
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    # Rotate the tick labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add text annotations
    thresh = cm.max() / 2.0
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14,
            )

    ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=12, fontweight="bold")
    ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✓ Confusion matrix saved as 'confusion_matrix.png'\n")

In [44]:
def print_classification_report(y_true, y_pred, classes=["Cat (0)", "Dog (1)"]):
    """Print and save classification report."""
    print("=" * 80)
    print("CLASSIFICATION REPORT ON TEST DATASET")
    print("=" * 80)

    report = classification_report(y_true, y_pred, target_names=classes, digits=4)
    print(report)

    # Save to file
    with open("classification_report.txt", "w") as f:
        f.write("CLASSIFICATION REPORT ON TEST DATASET\n")
        f.write("=" * 80 + "\n")
        f.write(report)

    print("=" * 80)
    print("✓ Classification report saved as 'classification_report.txt'\n")

In [ ]:
def fgsm_attack(image, epsilon, data_grad):
    """
    Perform FGSM attack on a single image.

    Args:
        image: Original input image tensor
        epsilon: Perturbation magnitude
        data_grad: Gradient of loss w.r.t. input image

    Returns:
        Perturbed image
    """
    # Get the sign of the gradient
    sign_data_grad = data_grad.sign()

    # Create the perturbed image by adjusting in the direction that increases loss
    perturbed_image = image + epsilon * sign_data_grad

    # Clip to maintain valid image range [0, 1] after denormalization
    # For normalized images, we keep them in the normalized range
    perturbed_image = torch.clamp(perturbed_image, image.min().item(), image.max().item())

    return perturbed_image


def generate_adversarial_examples(model, device, test_loader, epsilon, class_names=["Cat", "Dog"]):
    """
    Generate adversarial examples for the entire test set using FGSM.

    Args:
        model: Trained model
        device: torch device
        test_loader: DataLoader for test set
        epsilon: Perturbation strength
        class_names: List of class names

    Returns:
        Dictionary with accuracy, examples, and statistics
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()

    correct = 0
    total = 0
    adv_examples = []

    # Store first few examples for visualization
    max_examples = 5
    examples_collected = 0

    print(f"\nGenerating adversarial examples with epsilon={epsilon}...")

    for data, target in tqdm(test_loader, desc=f"ε={epsilon}"):
        data, target = data.to(device), target.to(device)

        # Set requires_grad to True for input data
        data.requires_grad = True

        # Forward pass
        output = model(data)
        init_pred = output.argmax(dim=1)

        # Calculate loss
        loss = criterion(output, target)

        # Zero all gradients
        model.zero_grad()

        # Calculate gradients w.r.t. input
        loss.backward()

        # Get gradient of loss w.r.t. data
        data_grad = data.grad.data

        # Generate perturbed image
        perturbed_data = fgsm_attack(data, epsilon, data_grad)

        # Re-classify the perturbed image
        output = model(perturbed_data)
        final_pred = output.argmax(dim=1)

        # Check accuracy
        correct += (final_pred == target).sum().item()
        total += target.size(0)

        # Save some examples for visualization
        if examples_collected < max_examples:
            for i in range(data.size(0)):
                if examples_collected >= max_examples:
                    break

                # Store original, adversarial, and perturbation
                adv_examples.append(
                    {
                        "original": data[i].cpu().detach(),
                        "adversarial": perturbed_data[i].cpu().detach(),
                        "perturbation": (perturbed_data[i] - data[i]).cpu().detach(),
                        "true_label": target[i].item(),
                        "original_pred": init_pred[i].item(),
                        "adversarial_pred": final_pred[i].item(),
                        "epsilon": epsilon,
                    }
                )
                examples_collected += 1

    # Calculate final accuracy
    accuracy = 100 * correct / total

    print(f"Epsilon: {epsilon}")
    print(f"Test Accuracy: {accuracy:.2f}% ({correct}/{total})")
    print(f"Attack Success Rate: {100 - accuracy:.2f}%")

    return {"epsilon": epsilon, "accuracy": accuracy, "correct": correct, "total": total, "examples": adv_examples}


def visualize_adversarial_examples(results_list, class_names=["Cat", "Dog"]):
    """
    Visualize adversarial examples for different epsilon values.

    Args:
        results_list: List of results dictionaries from generate_adversarial_examples
        class_names: List of class names
    """
    num_epsilons = len(results_list)
    num_examples = len(results_list[0]["examples"])

    # Denormalization function
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def denormalize(img):
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        return img.permute(1, 2, 0).numpy()

    # Create figure with subplots
    fig, axes = plt.subplots(num_examples, num_epsilons * 3, figsize=(num_epsilons * 9, num_examples * 3))

    if num_examples == 1:
        axes = axes.reshape(1, -1)

    for row_idx in range(num_examples):
        for eps_idx, results in enumerate(results_list):
            example = results["examples"][row_idx]
            epsilon = results["epsilon"]

            col_base = eps_idx * 3

            # Original image
            orig_img = denormalize(example["original"])
            axes[row_idx, col_base].imshow(orig_img)
            axes[row_idx, col_base].set_title(
                f"Original\n{class_names[example['true_label']]}\nPred: {class_names[example['original_pred']]}",
                fontsize=10,
            )
            axes[row_idx, col_base].axis("off")

            # Perturbation (amplified for visibility)
            perturbation = example["perturbation"]
            pert_img = perturbation.permute(1, 2, 0).numpy()
            # Amplify and center around 0.5 for visualization
            pert_img = (pert_img - pert_img.min()) / (pert_img.max() - pert_img.min() + 1e-8)
            axes[row_idx, col_base + 1].imshow(pert_img, cmap="RdBu", vmin=0, vmax=1)
            axes[row_idx, col_base + 1].set_title(f"Perturbation\nε={epsilon}\n(Amplified)", fontsize=10)
            axes[row_idx, col_base + 1].axis("off")

            # Adversarial image
            adv_img = denormalize(example["adversarial"])
            axes[row_idx, col_base + 2].imshow(adv_img)

            # Mark if attack succeeded
            attack_success = example["original_pred"] != example["adversarial_pred"]
            color = "red" if attack_success else "green"
            axes[row_idx, col_base + 2].set_title(
                f"Adversarial\n{class_names[example['true_label']]}\nPred: {class_names[example['adversarial_pred']]}",
                fontsize=10,
                color=color,
                fontweight="bold",
            )
            axes[row_idx, col_base + 2].axis("off")

    plt.tight_layout()
    plt.savefig("adversarial_examples.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_accuracy_vs_epsilon(results_list):
    """
    Plot accuracy as a function of epsilon.

    Args:
        results_list: List of results dictionaries
    """
    epsilons = [r["epsilon"] for r in results_list]
    accuracies = [r["accuracy"] for r in results_list]

    plt.figure(figsize=(10, 6))
    plt.plot(epsilons, accuracies, "b-o", linewidth=2, markersize=8)
    plt.xlabel("Epsilon (Perturbation Strength)", fontsize=12)
    plt.ylabel("Test Accuracy (%)", fontsize=12)
    plt.title("Model Robustness vs Adversarial Perturbation Strength", fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.3)

    # Add accuracy values on points
    for eps, acc in zip(epsilons, accuracies, strict=False):
        plt.annotate(f"{acc:.1f}%", xy=(eps, acc), xytext=(5, 5), textcoords="offset points", fontsize=10)

    plt.tight_layout()
    plt.savefig("accuracy_vs_epsilon.png", dpi=150, bbox_inches="tight")
    plt.show()


def analyze_adversarial_robustness(
    model, test_loader, device, epsilons=[0, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3], class_names=["Cat", "Dog"]
):
    """
    Complete adversarial robustness analysis.

    Args:
        model: Trained model
        test_loader: Test data loader
        device: torch device
        epsilons: List of epsilon values to test
        class_names: List of class names
    """
    print("\n" + "=" * 80)
    print("ADVERSARIAL ROBUSTNESS ANALYSIS - FGSM ATTACK")
    print("=" * 80)

    print(f"\nTesting {len(epsilons)} different epsilon values: {epsilons}")
    print("Epsilon (ε) controls the magnitude of adversarial perturbation")
    print("  • ε = 0: No perturbation (baseline accuracy)")
    print("  • Small ε: Subtle, imperceptible changes")
    print("  • Large ε: More visible perturbations, stronger attack")

    results_list = []

    # Test each epsilon value
    for epsilon in epsilons:
        results = generate_adversarial_examples(model, device, test_loader, epsilon, class_names)
        results_list.append(results)
        print()

    # Print summary table
    print("\n" + "=" * 80)
    print("SUMMARY OF RESULTS")
    print("=" * 80)
    print(f"{'Epsilon':<12} {'Accuracy':<12} {'Attack Success Rate':<20} {'Status'}")
    print("-" * 80)

    for results in results_list:
        eps = results["epsilon"]
        acc = results["accuracy"]
        asr = 100 - acc

        if eps == 0:
            status = "Baseline (No Attack)"
        elif acc > 80:
            status = "🟢 Robust"
        elif acc > 50:
            status = "🟡 Vulnerable"
        else:
            status = "🔴 Very Vulnerable"

        print(f"{eps:<12.3f} {acc:<12.2f} {asr:<20.2f} {status}")

    print("=" * 80)

    # Visualize examples
    print("\nGenerating visualizations...")

    # Filter results to show 3 most interesting epsilon values (excluding 0)
    display_results = [r for r in results_list if r["epsilon"] > 0][:3]
    if results_list[0]["epsilon"] == 0:
        display_results = [results_list[0]] + display_results[:2]

    visualize_adversarial_examples(display_results, class_names)

    # Plot accuracy vs epsilon
    plot_accuracy_vs_epsilon(results_list)

    # Analysis and insights
    print("\n" + "=" * 80)
    print("KEY INSIGHTS")
    print("=" * 80)

    baseline_acc = results_list[0]["accuracy"] if results_list[0]["epsilon"] == 0 else None

    if baseline_acc:
        print(f"• Baseline Accuracy (ε=0): {baseline_acc:.2f}%")

    # Find epsilon where accuracy drops below 50%
    critical_eps = None
    for r in results_list:
        if r["epsilon"] > 0 and r["accuracy"] < 50:
            critical_eps = r["epsilon"]
            break

    if critical_eps:
        print(f"• Model becomes unreliable at ε ≥ {critical_eps:.3f} (accuracy < 50%)")
    else:
        print("• Model remains above 50% accuracy for all tested epsilon values")

    # Robustness analysis
    if baseline_acc:
        for r in results_list[1:4]:  # First 3 non-zero epsilons
            acc_drop = baseline_acc - r["accuracy"]
            print(
                f"• At ε={r['epsilon']:.3f}: Accuracy drops by {acc_drop:.2f}% "
                f"({baseline_acc:.2f}% → {r['accuracy']:.2f}%)"
            )
    print("\nAdversarial analysis complete!")
    print("Saved visualizations:")
    print("  - adversarial_examples.png")
    print("  - accuracy_vs_epsilon.png")

    return results_list

In [ ]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 2

# Control flags
FORCE_RETRAIN = False  # Set to True to retrain even if model exists
MODEL_PATH = "best_model.pth"

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")


# ============================================================================
# HELPER: Check if we should skip training
# ============================================================================
def should_skip_training():
    """Check if model exists and we shouldn't force retrain."""
    if FORCE_RETRAIN:
        print("⚠️ FORCE_RETRAIN is True - will retrain model")
        return False

    if os.path.exists(MODEL_PATH):
        print(f"Found existing model at '{MODEL_PATH}'")
        print("  Set FORCE_RETRAIN=True to retrain")
        return True

    print("✗ No existing model found - will train")
    return False


# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("=" * 80)
print("DATA LOADING")
print("=" * 80)
print("Loading dataset")
data = load_dataset("pantelism/cats-vs-dogs")

print("Preparing images...")
train_images = [img for img in data["train"]["image"]]
train_labels = np.array(data["train"]["label"])

print(f"Total images: {len(train_images)}")
print(f"Label distribution: {np.unique(train_labels, return_counts=True)}\n")

# Split into train, validation, and test
train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    train_images, train_labels, test_size=0.3, random_state=42, stratify=train_labels
)

val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42, stratify=temp_lbls
)

print(f"\nTrain set: {len(train_imgs)} images")
print(f"Validation set: {len(val_imgs)} images")
print(f"Test set: {len(test_imgs)} images\n")

# ============================================================================
# 2. CREATE DATASETS AND DATALOADERS
# ============================================================================
print("Creating datasets...")

# Data transforms
train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Create datasets and dataloaders
train_dataset = CatsDogsDataset(train_imgs, train_lbls, transform=train_transform)
val_dataset = CatsDogsDataset(val_imgs, val_lbls, transform=val_transform)
test_dataset = CatsDogsDataset(test_imgs, test_lbls, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("✓ Datasets ready!\n")

# ============================================================================
# 3. CREATE MODEL
# ============================================================================
model = SimpleCNN(num_classes=NUM_CLASSES)
model = model.to(device)

# Print model summary, must print summary or else it does not show on notebooks
print(model)
print("\n" + "=" * 80)
print("MODEL SUMMARY")
print("=" * 80)
print(
    summary(
        model,
        input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE),
        col_names=["input_size", "output_size", "num_params"],
        depth=3,
    )
)
print("=" * 80 + "\n")

# ============================================================================
# 4. TRAIN THE MODEL (OR SKIP IF MODEL EXISTS)
# ============================================================================
if should_skip_training():
    print("\nSKIPPING TRAINING - Loading existing model\n")
    model.load_state_dict(torch.load(MODEL_PATH))
    print(f"Model loaded from '{MODEL_PATH}'\n")

else:
    print("\n" + "=" * 80)
    print("TRAINING MODEL")
    print("=" * 80)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    print("Starting training...\n")
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print("-" * 40)

        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)
            print("Model saved!")
        print()

    print(f"Training complete! Best validation accuracy: {best_val_acc:.2f}%\n")

    # Plot training history
    plot_training_history(history)

    # Load best model
    model.load_state_dict(torch.load(MODEL_PATH))

# ============================================================================
# 5. EVALUATE ON TEST SET
# ============================================================================
print("\n" + "=" * 80)
print("TEST SET EVALUATION")
print("=" * 80)

criterion = nn.CrossEntropyLoss()

print("Evaluating on test set...")
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}\n")

# Confusion Matrix
print("Generating predictions...")
y_pred, y_true = get_predictions(model, test_loader, device)
plot_confusion_matrix(y_true, y_pred, classes=["Cat (0)", "Dog (1)"])

# Classification Report
print_classification_report(y_true, y_pred, classes=["Cat (0)", "Dog (1)"])

# ========================================================================
# SUMMARY
# ========================================================================

print(f"\nFinal Test Accuracy: {test_acc:.2f}%")
print("=" * 80)

# TASK 3

# Load best model
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# Test adversarial robustness with different epsilon values
epsilons = [0, 0.01, 0.05, 0.1]
results = analyze_adversarial_robustness(model, test_loader, device, epsilons=epsilons, class_names=["Cat", "Dog"])